In [ ]:
%cd YOUR_PATH_HERE

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from model import UNET
from utils import get_loaders, check_accuracy, save_checkpoint
from train import train_fn
from monai.losses import DiceLoss, DiceCELoss, GeneralizedDiceLoss, FocalLoss
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import logging
import os
from concurrent.futures import ThreadPoolExecutor
import hashlib
import sqlite3

pruner = optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=5, interval_steps=1)

logging.basicConfig(level=logging.INFO)

CHECKPOINT_DIR = "checkpoints/"
SQLITE_DB_PATH = "sqlite:///example.db"

db_path = "example.db"
if os.path.exists(db_path):
    print(f"The file {db_path} exists.")
else:
    print(f"The file {db_path} doesn't exist.")

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

checkpoints = os.listdir(CHECKPOINT_DIR)
if checkpoints:
    print("Checkpoints found:")
    for checkpoint in checkpoints:
        print(checkpoint)
else:
    print("No checkpoint found.")

logging.info(f"Using SQLite database at {SQLITE_DB_PATH}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_IMG_DIR = "data/train_images"
TRAIN_MASK_DIR = "data/train_masks"
VAL_IMG_DIR = "data/val_images"
VAL_MASK_DIR = "data/val_masks"
NUM_WORKERS = 20
PIN_MEMORY = True

def to_float32(image, **kwargs):
    return image.astype(np.float32)

def save_checkpoint_async(state, filename):
    with ThreadPoolExecutor() as executor:
        future = executor.submit(torch.save, state, filename)
    try:
        future.result()
        logging.info(f"Checkpoint saved successfully to {filename}")
    except Exception as e:
        logging.error(f"Error saving checkpoint to {filename}: {e}")
    return future

def generate_trial_dir(trial):
    lr = trial.params.get('lr')
    batch_size = trial.params.get('batch_size')
    features_option = trial.params.get('features')
    loss_function_name = trial.params.get('loss_function')

    config_str = f"{lr}_{batch_size}_{features_option}_{loss_function_name}"
    hash_object = hashlib.md5(config_str.encode())
    trial_dir = hash_object.hexdigest()
    print(f"Generated trial directory: {trial_dir}")
    return os.path.join(CHECKPOINT_DIR, trial_dir)

def load_checkpoint(checkpoint_path, model, optimizer):
    print(f"Attempting to load checkpoint from {checkpoint_path}")
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        logging.info(f"Checkpoint loaded from {checkpoint_path}")
        return checkpoint.get('epoch', 0), checkpoint.get('best_dice_score', 0.0)
    else:
        logging.warning(f"Checkpoint {checkpoint_path} not found")
        return 0, 0.0

def find_last_trial(study):
    if not study.trials:
        return None

    last_trial = max(study.trials, key=lambda t: t.number)
    return last_trial

def print_best_trial(study, trial):
    if len(study.trials) > 0 and any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
        best_trial = study.best_trial
        print('Best trial so far:')
        print(f'  Value: {best_trial.value}')
        print('  Params: ')
        for key, value in best_trial.params.items():
            print(f'    {key}: {value}')
    else:
        print("No completed trials found.")

def objective(trial, start_epoch=0, best_dice_score=0.0, checkpoint_path=None):

    lr = 0.00019218433499427676
    batch_size = 3
    num_epochs = trial.suggest_int('num_epochs', 5, 35)  
    features = (64, 128, 256, 512)
    loss_function_name = 'BCEWithLogitsLoss'
    dropout_rate = 0.21281137124418054

    alpha = trial.suggest_float('alpha', 1.0, 30.0)
    sigma = trial.suggest_float('sigma', 5.0, 15.0)

    brightness_contrast_p = trial.suggest_float('brightness_contrast_p', 0.1, 0.3)

    blur_limit_min = trial.suggest_int('blur_limit_min', 1, 7, step=2)
    blur_limit_max = trial.suggest_int('blur_limit_max', blur_limit_min + 2, 11, step=2)

    flip_p = trial.suggest_categorical('flip_p', [0.0, 0.5])

    rotate_limit = trial.suggest_int('rotate_limit', 5, 30)

    print(f'Trial {trial.number}:')
    print(f'  Learning Rate: {lr}')
    print(f'  Batch Size: {batch_size}')
    print(f'  Number of Epochs: {num_epochs}')
    print(f'  Features: {features}')
    print(f'  Loss Function: {loss_function_name}')
    print(f'  Dropout Rate: {dropout_rate}')
    print(' ')

    print(f'  Rotate Limit: {rotate_limit}')

    print(f'  Horizontal Flip Probability: {flip_p}')

    print(f'  ElasticTransform Alpha: {alpha}')
    print(f'  ElasticTransform Sigma: {sigma}')

    print(f'  Blur Limit Min: {blur_limit_min}')
    print(f'  Blur Limit Max: {blur_limit_max}')

    if loss_function_name == 'BCEWithLogitsLoss':
        loss_fn = nn.BCEWithLogitsLoss()
    else:
        logging.error(f"Loss function '{loss_function_name}' not recognized.")
        return 0.0

    model = UNET(in_channels=3, out_channels=1, features=features, dropout_rate=dropout_rate).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler()

    train_transform = A.Compose(
        [
            A.Resize(height=192, width=192),
            A.Rotate(limit=rotate_limit, p=0.5),
            A.HorizontalFlip(p=flip_p),
            A.ElasticTransform(alpha=alpha, sigma=sigma, p=0.5),
            A.RandomBrightnessContrast(p=brightness_contrast_p),
            A.GaussianBlur(blur_limit=(blur_limit_min, blur_limit_max), p=0.1),
            A.Lambda(image=to_float32, mask=to_float32),
            A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0, always_apply=True),
            ToTensorV2(),
        ]
    )

    val_transform = A.Compose(
        [
            A.Resize(height=192, width=192),
            A.Lambda(image=to_float32, mask=to_float32),
            A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0, always_apply=True),
            ToTensorV2(),
        ]
    )

    train_loader, val_loader = get_loaders(
        TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR,
        batch_size, train_transform, val_transform, NUM_WORKERS, PIN_MEMORY
    )

    if checkpoint_path and os.path.exists(checkpoint_path):
        start_epoch, best_dice_score = load_checkpoint(checkpoint_path, model, optimizer)
        logging.info(f"Resuming training from epoch {start_epoch} with best dice score {best_dice_score}")
    else:
        logging.info(f"No valid checkpoint found at {checkpoint_path}, starting from scratch.")


    checkpoint_dir = generate_trial_dir(trial)
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pth.tar")

    for epoch in range(start_epoch, num_epochs):
        print(f"Starting epoch {epoch+1}/{num_epochs}")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        print(f"Memory allocated: {torch.cuda.memory_allocated(DEVICE) / 1024 ** 3:.2f} GB")
        print(f"Max memory allocated: {torch.cuda.max_memory_allocated(DEVICE) / 1024 ** 3:.2f} GB")
        print(f"Memory cached: {torch.cuda.memory_reserved(DEVICE) / 1024 ** 3:.2f} GB")

        print("Starting Dice calculation")
        current_dice_score = check_accuracy(val_loader, model, device=DEVICE)
        print(f"Epoch {epoch+1}/{num_epochs} - Dice Score: {current_dice_score}")

        logging.info(f"Trial {trial.number} - Epoch {epoch+1}/{num_epochs} - Dice Score: {current_dice_score}")

        trial.report(current_dice_score, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if current_dice_score > best_dice_score:
            best_dice_score = current_dice_score
            checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1,
                "best_dice_score": best_dice_score
            }
            future = save_checkpoint_async(checkpoint, checkpoint_path)
            print(f"Checkpoint saved with better Dice: {current_dice_score}")

    print(f'Trial {trial.number} returning Dice Score: {best_dice_score}')

    return best_dice_score

def print_all_trials(study):
    trials = study.trials
    for trial in trials:
        print(f"Trial {trial.number}:")
        print(f"  State: {trial.state}")
        print(f"  Value: {trial.value}")
        print(f"  Params: {trial.params}")

if __name__ == '__main__':
    logging.info("Starting Optuna study...")
    if os.path.exists("example.db"):
        logging.info("Found existing SQLite database, resuming study...")
    else:
        logging.info("No existing SQLite database found, starting new study...")

    study_name = "optuna_study"
    study = optuna.create_study(study_name=study_name, direction='maximize', storage=SQLITE_DB_PATH, load_if_exists=True, pruner=pruner)

    completed_or_pruned_trials = [t for t in study.trials if t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]]
    num_completed_or_pruned_trials = len(completed_or_pruned_trials)
    total_trials = 100
    remaining_trials = total_trials - num_completed_or_pruned_trials

    logging.info(f"{num_completed_or_pruned_trials} trials already completed or pruned. {remaining_trials} trials remaining.")

    last_trial = find_last_trial(study)
    if last_trial:
        if last_trial.state == optuna.trial.TrialState.RUNNING:
            print(f"Resuming trial {last_trial.number}")
            try:
                checkpoint_dir = generate_trial_dir(last_trial)
                checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pth.tar")
                print(f"Checkpoint path: {checkpoint_path}")

                features_option = last_trial.params['features']
                features = tuple(map(int, features_option.split(',')))
                dropout_rate = last_trial.params['dropout_rate']
                model = UNET(in_channels=3, out_channels=1, features=features, dropout_rate=dropout_rate).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=last_trial.params['lr'])

                start_epoch, best_dice_score = load_checkpoint(checkpoint_path, model, optimizer)
                objective(last_trial, start_epoch=start_epoch, best_dice_score=best_dice_score, checkpoint_path=checkpoint_path)

                completed_or_pruned_trials = [t for t in study.trials if t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]]
                num_completed_or_pruned_trials = len(completed_or_pruned_trials)
                remaining_trials = total_trials - num_completed_or_pruned_trials
            except Exception as e:
                logging.error(f"Error in trial {last_trial.number}: {e}")

        else:
            logging.info(f"Last trial {last_trial.number} was completed or pruned. Starting new trials from {last_trial.number + 1}.")
            try:
                study.optimize(objective, n_trials=remaining_trials, timeout=None, callbacks=[print_best_trial])
                remaining_trials = 0 
            except Exception as e:
                logging.error(f"Optimization interrupted: {e}")

    if remaining_trials > 0:
        try:
            study.optimize(objective, n_trials=remaining_trials, timeout=None, callbacks=[print_best_trial])
        except Exception as e:
            logging.error(f"Optimization interrupted: {e}")
    elif not last_trial:
        logging.info("No trials found. Starting fresh study.")
        try:
            study.optimize(objective, n_trials=total_trials, timeout=None, callbacks=[print_best_trial])
        except Exception as e:
            logging.error(f"Optimization interrupted: {e}")

    if len(study.trials) > 0 and any(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials):
        print('Best trial:')
        trial = study.best_trial

        print(f'  Value: {trial.value}')
        print('  Params: ')
        for key, value in trial.params.items():
            print(f'    {key}: {value}')
    else:
        print("No completed trials found.")

    print_all_trials(study)